# Step 2 — Map SullyGnome Twitch statistics to Steam app IDs

**Input:**

- `data/sullygnome/monthly/YYYY-MM.csv` — the top 2,000 Twitch game categories per month (Jan 2017 – Dec 2020), scraped from SullyGnome.
- `data/sullygnome/appid_map.csv` — candidate links between a SullyGnome game id and a Steam `appid`. These were matched on the Twitch category URL slug (`via = url`) or on the exact game name (`via = name`).

**Output:** `data/sullygnome/twitch_monthly_by_appid.csv` — one row per Steam app and month.

The candidate map is not trusted blindly. This notebook:

1. scores every link by name similarity and prints the weakest ones for manual review;
2. removes links found wrong in that review;
3. sums Twitch categories that belong to the same Steam game (e.g. a category that was renamed), so one game never gets two rows for the same month.

In [1]:
from difflib import SequenceMatcher
from pathlib import Path
import re

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
SULLY_DIR = PROJECT_ROOT / "data" / "sullygnome"

mapping = pd.read_csv(SULLY_DIR / "appid_map.csv")
monthly = pd.concat([
    pd.read_csv(path).assign(month=re.search(r"\d{4}-\d{2}", path.name).group(0))
    for path in sorted((SULLY_DIR / "monthly").glob("*.csv"))
], ignore_index=True)
print(f"Candidate links: {len(mapping):,} ({mapping['via'].value_counts().to_dict()})")
print(f"Monthly Twitch rows: {len(monthly):,} across {monthly['month'].nunique()} months")

Candidate links: 1,237 ({'url': 1189, 'name': 48})
Monthly Twitch rows: 96,000 across 48 months


## 1. Review the weakest links

A low similarity score is not proof of an error: Twitch often uses short names ("FNaF 1", "PUBG: BATTLEGROUNDS"). Every link scoring under 0.6 was checked by hand.

In [2]:
mapping["name_similarity"] = [
    SequenceMatcher(None, str(steam).lower(), str(sully).lower()).ratio()
    for steam, sully in zip(mapping["name_steam"], mapping["name_sully"])
]
weak = mapping.sort_values("name_similarity").query("name_similarity < 0.6")
weak[["appid", "name_steam", "name_sully", "via", "name_similarity"]].round(2)

,appid,name_steam,name_sully,via,name_similarity
663,319510,Five Nights at Freddy's,FNaF 1,url,0.14
1112,561600,MXGP3 - The Official Motocross Videogame,MXGP 3,url,0.22
152,38400,Fallout,Fallout: A Post Nuclear Role Playing Game,url,0.29
961,451340,Gold Rush: The Game,Gold Mining Simulator,url,0.30
681,323580,Jotun,Jotun: Valhalla Edition,url,0.36
716,335670,LISA,LISA: The Painful,url,0.38
1099,552920,Sailaway - The Sailing Simulator,Sailaway,url,0.40
328,222880,Insurgency,INSURGENCY: Modern Infantry Combat,url,0.45
1125,570780,????(GuJian),GuJian Online,url,0.48
731,345390,King's Quest,King's Quest: Quest for the Crown,url,0.53


Review outcome: most weak links are correct short names (FNaF 1, MXGP 3, Fallout, Jotun, LISA, Sailaway, Fate/Extella). Five point to a **different game**. A sixth link with a high score (Jackbox Party Pack 3 → "Jackbox Party Packs") is an **umbrella category** shared by all packs. These six are removed.

In [3]:
REJECTED_LINKS = {
    113020: "Monaco (2013) matched to Monaco Pinball",
    222880: "Insurgency (2014) matched to the 2007 Source mod Insurgency: Modern Infantry Combat",
    345390: "King's Quest (2015) matched to King's Quest: Quest for the Crown (1984)",
    434170: "Jackbox Party Pack 3 matched to the umbrella category Jackbox Party Packs",
    451340: "Gold Rush: The Game matched to Gold Mining Simulator",
    570780: "GuJian (single-player) matched to GuJian Online",
}
clean_mapping = mapping[~mapping["appid"].isin(REJECTED_LINKS)]
print(f"Removed {len(mapping) - len(clean_mapping)} wrong links; {len(clean_mapping):,} remain")

shared = clean_mapping[clean_mapping["appid"].duplicated(keep=False)].sort_values("appid")
print(f"Steam apps linked to more than one Twitch category: {shared['appid'].nunique()}")
shared[["appid", "name_steam", "sully_id", "name_sully", "via"]]

Removed 6 wrong links; 1,231 remain
Steam apps linked to more than one Twitch category: 13


,appid,name_steam,sully_id,name_sully,via
98,17460,Mass Effect,397,Mass Effect,url
99,17460,Mass Effect,68337,Mass Effect,name
146,34270,SEGA Mega Drive & Genesis Classics,41288,SEGA Mega Drive & Genesis Classics,name
147,34270,SEGA Mega Drive & Genesis Classics,32593,SEGA Mega Drive & Genesis Classics,url
251,203160,Tomb Raider,176,Tomb Raider,url
252,203160,Tomb Raider,31980,Tomb Raider,name
334,224060,Deadpool,510,Deadpool,url
335,224060,Deadpool,46081,DEADPOOL,url
402,239120,FINAL FANTASY III,11504,FINAL FANTASY III,url
403,239120,FINAL FANTASY III,60923,Final Fantasy III,url


## 2. Build the monthly Twitch table per Steam app

Where one Steam game has several Twitch categories (renamed or duplicated categories), the monthly values are summed. Average viewers, view minutes and channels add up across categories; peak viewers takes the maximum, which is a lower bound.

In [4]:
TWITCH_COLUMNS = ["avgviewers", "viewminutes", "avgchannels", "uniquechannels"]
twitch = (
    monthly.merge(clean_mapping[["appid", "sully_id", "name_steam"]], left_on="id", right_on="sully_id")
    .groupby(["appid", "month"], as_index=False)
    .agg(name_steam=("name_steam", "first"), maxviewers=("maxviewers", "max"),
         **{column: (column, "sum") for column in TWITCH_COLUMNS})
    .sort_values(["appid", "month"])
    .reset_index(drop=True)
)
assert not twitch[["appid", "month"]].duplicated().any()
print(f"Twitch rows: {len(twitch):,} | Steam apps with any Twitch data: {twitch['appid'].nunique():,}")
print(f"Smallest monthly average audience that still makes the top 2,000: "
      f"{monthly.groupby('month')['avgviewers'].min().median():.0f} viewers")

twitch.to_csv(SULLY_DIR / "twitch_monthly_by_appid.csv", index=False)
clean_mapping.drop(columns="name_similarity").to_csv(SULLY_DIR / "appid_map_validated.csv", index=False)
print("Wrote twitch_monthly_by_appid.csv and appid_map_validated.csv")

Twitch rows: 28,125 | Steam apps with any Twitch data: 1,218
Smallest monthly average audience that still makes the top 2,000: 5 viewers
Wrote twitch_monthly_by_appid.csv and appid_map_validated.csv


Because each monthly file lists only the top 2,000 categories, a mapped game that is missing in a month was **below the top-2,000 cut-off**, not necessarily at zero viewers. The analysis therefore uses 0 for such months together with an explicit `twitch_mapped` flag. A game that was never mapped is kept apart from a game with near-zero Twitch attention.